# 🏗️ Verificação de Escadas — NBR 9077 + NBR 9050
## Sistema Multi-Agente CrewAI + IfcOpenShell

**Projeto:** Residência Rua das Palmeiras 142 — Recife/PE  
**Normas:** NBR 9077:2001 (Saídas de Emergência) + NBR 9050:2020 (Acessibilidade)  
**Framework:** CrewAI 1.x + IfcOpenShell + OpenAI GPT-4o-mini

---
### ▶️ Como usar
1. Vá em **Runtime → Run all**
2. Cole sua **API Key da OpenAI** na Célula 2
3. Aguarde ~3-5 minutos — os agentes rodam em sequência
4. Baixe o `.docx` e `.xlsx` gerados no final

---
### 🔍 O que este sistema verifica

| Critério | Norma | Parâmetro |
|---|---|---|
| Altura do espelho (degrau) | NBR 9077 §5.2 | máx. 18 cm |
| Profundidade do cobertor (degrau) | NBR 9077 §5.2 | mín. 28 cm |
| Largura da escada | NBR 9077 §5.2.2 | mín. 1,20 m |
| Número máx. de degraus por lance | NBR 9077 §5.2.3 | máx. 18 |
| Dimensões do patamar | NBR 9077 §5.2.4 | larg. ≥ escada; prof. ≥ 1,20 m |
| Largura da porta de acesso | NBR 9050 §4.6.3 / NBR 9077 | mín. 0,90 m |
| Altura da porta de acesso | NBR 9050 §4.6.3 | mín. 2,10 m |
| Sentido de abertura da porta | NBR 9077 §5.4 | abre p/ fora da caixa |


## Celula 1 - Instalacao de dependencias

> Instalando as bibliotecas em silencio - nao vai aparecer nada por 1-2 minutos, isso e normal! O pip esta instalando: IfcOpenShell, CrewAI, python-docx e openpyxl. No final aparecem duas linhas confirmando as versoes - se aparecerem, tudo certo!

In [32]:
print('⏳ Instalando dependências...')

!pip install ifcopenshell -q
!pip install crewai crewai-tools -q
!pip install python-docx openpyxl -q

import ifcopenshell, crewai
print(f'\n✅ IfcOpenShell: {ifcopenshell.version}')
print(f'✅ CrewAI: {crewai.__version__}')


⏳ Instalando dependências...

✅ IfcOpenShell: 0.8.4.post1
✅ CrewAI: 1.13.0


## Celula 2 - Configuracao

> Aqui configuramos o modelo GPT-4o-mini, criamos a pasta de outputs e baixamos automaticamente o arquivo IFC do GitHub - sem precisar fazer upload manual.

In [33]:
# Download automático do IFC do GitHub
import os
if not os.path.exists("/content/palmeiras_142_enriquecido.ifc"):
    print("⬇️  Baixando arquivo IFC do GitHub...")
    !wget -q https://raw.githubusercontent.com/srosenboim/M5T2-Avaliacao-individual/main/palmeiras_142_enriquecido.ifc -O /content/palmeiras_142_enriquecido.ifc
    print("✅ IFC baixado!")
else:
    print("✅ IFC já presente em /content/")

import os
from pathlib import Path
from datetime import datetime

# ✏️ COLE SUA API KEY AQUI
os.environ['OPENAI_API_KEY'] = 'sk-COLE-SUA-CHAVE-AQUI'

LLM_MODEL  = 'gpt-4o-mini'   # econômico; use 'gpt-4o' para maior precisão
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMESTAMP  = datetime.now().strftime('%Y%m%d_%H%M%S')

# Caminho do IFC — use o arquivo enriquecido fornecido no repositório
# Se quiser usar o seu próprio .ifc, altere o caminho abaixo:
IFC_PATH = '/content/palmeiras_142_enriquecido.ifc'

print(f'✅ Modelo LLM  : {LLM_MODEL}')
print(f'✅ Output dir  : {OUTPUT_DIR}')
print(f'✅ IFC path    : {IFC_PATH}')
key = os.environ.get('OPENAI_API_KEY','')
if 'COLE' in key:
    print('\n⚠️  Atenção: cole sua API Key acima!')
else:
    print(f'✅ API Key     : {key[:8]}...{key[-4:]}')


✅ IFC já presente em /content/
✅ Modelo LLM  : gpt-4o-mini
✅ Output dir  : /content/outputs
✅ IFC path    : /content/palmeiras_142_enriquecido.ifc
✅ API Key     : sk-proj-...0wQA


## Celula 3 - Extracao do IFC

> Aqui o IfcOpenShell abre o arquivo IFC e extrai os dados reais das escadas: espelho, cobertor, largura, numero de degraus, dimensoes dos patamares e portas de acesso. Nenhum LLM envolvido - dados puros do modelo BIM.

In [34]:
import ifcopenshell
import ifcopenshell.util.element
import json

def extrair_dados_ifc(ifc_path: str) -> dict:
    """
    Extrai dados de escadas, patamares e portas de acesso do arquivo IFC.
    Usa IfcOpenShell — leitura determinística, sem LLM.

    Retorna dicionário com:
      - project_name: nome do projeto
      - storeys: lista de pavimentos
      - stairs: lista de escadas com Pset_StairCommon
      - stair_flights: lances com NumberOfRisers, RiserHeight, TreadLength
      - landings: patamares com largura e profundidade
      - access_doors: portas associadas às escadas
    """
    model = ifcopenshell.open(ifc_path)

    def get_props(el):
        """Extrai todas as propriedades de um elemento via Psets."""
        props = {}
        for rel in getattr(el, 'IsDefinedBy', []):
            if rel.is_a('IfcRelDefinesByProperties'):
                pset = rel.RelatingPropertyDefinition
                if hasattr(pset, 'HasProperties'):
                    for p in pset.HasProperties:
                        if hasattr(p, 'NominalValue') and p.NominalValue:
                            props[p.Name] = p.NominalValue.wrappedValue
        return props

    def get_storey_name(el):
        """Retorna o nome do pavimento que contém o elemento."""
        for rel in getattr(el, 'ContainedInStructure', []):
            if rel.is_a('IfcRelContainedInSpatialStructure'):
                return rel.RelatingStructure.Name or 'Não identificado'
        return 'Não identificado'

    # Pavimentos
    storeys = sorted([
        {'id': s.GlobalId, 'name': s.Name or 'Sem nome',
         'elevation': float(s.Elevation) if s.Elevation else 0.0}
        for s in model.by_type('IfcBuildingStorey')
    ], key=lambda x: x['elevation'])

    # Escadas (IfcStair) — dados gerais + Pset_StairCommon
    stairs = []
    for st in model.by_type('IfcStair'):
        p = get_props(st)
        stairs.append({
            'id':            st.GlobalId,
            'name':          st.Name or 'Sem nome',
            'description':   st.Description or '',
            'storey':        get_storey_name(st),
            'shape_type':    str(st.ShapeType) if hasattr(st,'ShapeType') else 'NOTDEFINED',
            # Pset_StairCommon
            'width':         p.get('Width'),
            'riser_height':  p.get('RiserHeight'),
            'tread_length':  p.get('TreadLength'),
            'n_risers':      p.get('NumberOfRiser'),
            'fire_exit':     p.get('FireExit'),
            'headroom':      p.get('RequiredHeadroom'),
        })

    # Lances (IfcStairFlight) — dados precisos por lance
    flights = []
    for lf in model.by_type('IfcStairFlight'):
        p = get_props(lf)
        flights.append({
            'id':           lf.GlobalId,
            'name':         lf.Name or 'Sem nome',
            'storey':       get_storey_name(lf),
            # Atributos do Pset_StairFlightCommon
            'n_risers':     p.get('NumberOfRiser'),  # Corrigido para usar Pset
            'n_treads':     p.get('NumberOfTreads'),  # Corrigido para usar Pset
            'riser_height': p.get('RiserHeight'),     # Corrigido para usar Pset
            'tread_length': p.get('TreadLength'),     # Corrigido para usar Pset
        })

    # Patamares (IfcSlab com PredefinedType=LANDING)
    landings = []
    for sl in model.by_type('IfcSlab'):
        if str(getattr(sl, 'PredefinedType', '')) == 'LANDING':
            p = get_props(sl)
            landings.append({
                'id':        sl.GlobalId,
                'name':      sl.Name or 'Sem nome',
                'storey':    get_storey_name(sl),
                'width':     p.get('Width'),
                'length':    p.get('Length'),
                'thickness': p.get('Thickness'),
                'area':      p.get('GrossArea'),
            })

    # Portas de acesso às escadas (contêm 'ESC' no nome)
    access_doors = []
    for d in model.by_type('IfcDoor'):
        if 'ESC' in (d.Name or ''):
            p = get_props(d)
            access_doors.append({
                'id':            d.GlobalId,
                'name':          d.Name or 'Sem nome',
                'storey':        get_storey_name(d),
                'width':         p.get('Width'),
                'height':        p.get('Height'),
                'operation_type':p.get('OperationType'),
                'fire_exit':     p.get('FireExit'),
            })

    project = model.by_type('IfcProject')
    project_name = project[0].Name if project else 'Projeto IFC'

    return {
        'project_name': project_name,
        'ifc_schema':   model.schema,
        'storeys':      storeys,
        'stairs':       stairs,
        'stair_flights':flights,
        'landings':     landings,
        'access_doors': access_doors,
    }


# ── Executar extração ─────────────────────────────────────────
ifc_data = extrair_dados_ifc(IFC_PATH)

print(f"\n📊 DADOS EXTRAÍDOS — {ifc_data['project_name']}")
print('='*55)
print(f"Schema  : {ifc_data['ifc_schema']}")
print(f"Escadas : {len(ifc_data['stairs'])}")
print(f"Lances  : {len(ifc_data['stair_flights'])}")
print(f"Patamares: {len(ifc_data['landings'])}")
print(f"Portas  : {len(ifc_data['access_doors'])}")
print()

for st in ifc_data['stairs']:
    print(f"  🪜 {st['name']:12s} | larg={st['width']}m | "
          f"espelho={st['riser_height']}m | cobertor={st['tread_length']}m | "
          f"degraus={int(st['n_risers'] or 0)}")

print()
for d in ifc_data['access_doors']:
    print(f"  🚪 {d['name']:22s} | larg={d['width']}m | "
          f"abertura={d['operation_type']}")

print()
for l in ifc_data['landings']:
    print(f"  📐 {l['name']:22s} | larg={l['width']}m | prof={l['length']}m")


📊 DADOS EXTRAÍDOS — Residência Rua das Palmeiras 142 — Recife/PE
Schema  : IFC2X3
Escadas : 2
Lances  : 2
Patamares: 2
Portas  : 4

  🪜 ESC-01       | larg=1.2m | espelho=0.17m | cobertor=0.29m | degraus=17
  🪜 ESC-02       | larg=1.1m | espelho=0.21m | cobertor=0.23m | degraus=20

  🚪 Porta ESC-01 Terreo    | larg=1.0m | abertura=SINGLE_SWING_LEFT
  🚪 Porta ESC-01 1oPav     | larg=1.0m | abertura=SINGLE_SWING_LEFT
  🚪 Porta ESC-02 Terreo    | larg=0.8m | abertura=SINGLE_SWING_RIGHT
  🚪 Porta ESC-02 1oPav     | larg=0.8m | abertura=SINGLE_SWING_RIGHT

  📐 Patamar ESC-01         | larg=1.2m | prof=1.2m
  📐 Patamar ESC-02         | larg=1.1m | prof=0.9m


## Celula 4 - Ferramentas de verificacao

> Aqui criamos as 4 ferramentas Python que os agentes vao usar: verificar escadas (NBR 9077), verificar portas (NBR 9050), gerar checklist Excel e gerar laudo Word.

In [35]:
import json
from crewai.tools import tool
from docx import Document as DocxDocument
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# ── Parâmetros normativos ─────────────────────────────────────
NBR = {
    # NBR 9077:2001 §5.2 — Geometria dos degraus
    'espelho_max':      0.18,   # m — altura máxima do espelho
    'espelho_min':      0.16,   # m — altura mínima do espelho
    'cobertor_min':     0.28,   # m — profundidade mínima do cobertor
    # NBR 9077:2001 §5.2.2 — Largura da escada
    'largura_min':      1.20,   # m — largura mínima da escada
    # NBR 9077:2001 §5.2.3 — Número de degraus por lance
    'degraus_max_lance':18,     # máximo de degraus por lance
    'degraus_min_lance': 3,     # mínimo de degraus por lance
    # NBR 9077:2001 §5.2.4 — Patamar
    'patamar_prof_min': 1.20,   # m — profundidade mínima do patamar
    # NBR 9050:2020 §4.6.3 + NBR 9077 §5.4 — Portas de acesso
    'porta_larg_min':   0.90,   # m — largura mínima da porta de acesso
    'porta_alt_min':    2.10,   # m — altura mínima da porta
    # Sentidos que NÃO são aceitáveis (abrem para DENTRO da caixa de escada)
    'abertura_proibida': ['SINGLE_SWING_RIGHT', 'DOUBLE_SWING_RIGHT'],
}


# ═══════════════════════════════════════════════════════════════
# FERRAMENTA 1 — Verificar escadas (degraus + largura + patamares)
# ═══════════════════════════════════════════════════════════════
@tool('Verificar Escadas NBR 9077')
def verificar_escadas(ifc_data_json: str) -> str:
    """
    Verifica escadas contra NBR 9077:2001 (saídas de emergência).
    Critérios: espelho máx 18cm, cobertor mín 28cm, largura mín 1,20m,
    máx 18 degraus/lance, patamar prof mín 1,20m.
    Input: JSON com dados IFC.
    Output: JSON com resultados de conformidade por escada.
    """
    data    = json.loads(ifc_data_json)
    stairs  = data.get('stairs', [])
    flights = data.get('stair_flights', [])
    landings= data.get('landings', [])
    results = []

    for st in stairs:
        issues, warnings, ok_items = [], [], []
        name = st['name']

        # Buscar lance correspondente
        lf = next((f for f in flights if st['name'].split('-')[1] in f['name']
                   if len(st['name'].split('-')) > 1), None)
        # Fallback: usar dados do Pset_StairCommon
        riser  = (lf['riser_height'] if lf and lf.get('riser_height') else st.get('riser_height'))
        tread  = (lf['tread_length'] if lf and lf.get('tread_length') else st.get('tread_length'))
        width  = st.get('width')
        n_ris  = int(lf['n_risers'] if lf and lf.get('n_risers') else st.get('n_risers') or 0)

        # 1. Espelho (altura do degrau)
        if riser is not None:
            rc = round(riser * 100, 1)
            if riser > NBR['espelho_max']:
                issues.append(
                    f"❌ Espelho: {rc}cm — excede máx. {NBR['espelho_max']*100:.0f}cm (NBR 9077 §5.2)")
            elif riser < NBR['espelho_min']:
                warnings.append(
                    f"⚠️ Espelho: {rc}cm — abaixo mín. {NBR['espelho_min']*100:.0f}cm (NBR 9077 §5.2)")
            else:
                ok_items.append(f"✅ Espelho: {rc}cm — conforme (16–18cm)")
        else:
            warnings.append('⚠️ Espelho: dado não encontrado no IFC')

        # 2. Cobertor (profundidade do degrau)
        if tread is not None:
            tc = round(tread * 100, 1)
            if tread < NBR['cobertor_min']:
                issues.append(
                    f"❌ Cobertor: {tc}cm — abaixo mín. {NBR['cobertor_min']*100:.0f}cm (NBR 9077 §5.2)")
            else:
                ok_items.append(f"✅ Cobertor: {tc}cm — conforme (mín. 28cm)")
        else:
            warnings.append('⚠️ Cobertor: dado não encontrado no IFC')

        # 3. Largura da escada
        if width is not None:
            wc = round(width * 100, 1)
            if width < NBR['largura_min']:
                issues.append(
                    f"❌ Largura: {wc}cm — abaixo mín. {NBR['largura_min']*100:.0f}cm (NBR 9077 §5.2.2)")
            else:
                ok_items.append(f"✅ Largura: {wc}cm — conforme (mín. 120cm)")
        else:
            warnings.append('⚠️ Largura: dado não encontrado no IFC')

        # 4. Número de degraus por lance
        if n_ris > 0:
            if n_ris > NBR['degraus_max_lance']:
                issues.append(
                    f"❌ Degraus/lance: {n_ris} — excede máx. {NBR['degraus_max_lance']} (NBR 9077 §5.2.3)")
            elif n_ris < NBR['degraus_min_lance']:
                issues.append(
                    f"❌ Degraus/lance: {n_ris} — abaixo mín. {NBR['degraus_min_lance']} (NBR 9077 §5.2.3)")
            else:
                ok_items.append(f"✅ Degraus/lance: {n_ris} — conforme (3–18)")
        else:
            warnings.append('⚠️ Número de degraus: dado não encontrado no IFC')

        # 5. Patamar correspondente
        # Associar pelo nome (ESC-01 → Patamar ESC-01)
        esc_id  = name  # ex: 'ESC-01'
        landing = next((l for l in landings if esc_id in l.get('name','')), None)
        if landing:
            lw = landing.get('width')
            ll = landing.get('length')
            if lw is not None and width is not None and lw < width:
                issues.append(
                    f"❌ Patamar largura: {round(lw*100)}cm < largura da escada {round(width*100)}cm (NBR 9077 §5.2.4)")
            elif lw is not None:
                ok_items.append(f"✅ Patamar largura: {round(lw*100)}cm — conforme")
            if ll is not None and ll < NBR['patamar_prof_min']:
                issues.append(
                    f"❌ Patamar profund.: {round(ll*100)}cm — abaixo mín. {NBR['patamar_prof_min']*100:.0f}cm (NBR 9077 §5.2.4)")
            elif ll is not None:
                ok_items.append(f"✅ Patamar profund.: {round(ll*100)}cm — conforme (mín. 120cm)")
        else:
            warnings.append('⚠️ Patamar: não identificado no IFC para esta escada')

        status = 'NÃO CONFORME' if issues else ('ATENÇÃO' if warnings else 'CONFORME')
        results.append({
            'element': name, 'id': st['id'], 'storey': st['storey'],
            'status': status,
            'issues': issues, 'warnings': warnings, 'ok_items': ok_items,
            'data': {
                'riser_height_m': riser, 'tread_length_m': tread,
                'width_m': width, 'n_risers': n_ris,
                'landing_width_m':  landing.get('width')  if landing else None,
                'landing_length_m': landing.get('length') if landing else None,
            }
        })

    summary = {
        'total':        len(results),
        'conforme':     sum(1 for r in results if r['status'] == 'CONFORME'),
        'nao_conforme': sum(1 for r in results if r['status'] == 'NÃO CONFORME'),
        'atencao':      sum(1 for r in results if r['status'] == 'ATENÇÃO'),
    }
    return json.dumps({'stairs_results': results, 'summary': summary}, ensure_ascii=False)


# ═══════════════════════════════════════════════════════════════
# FERRAMENTA 2 — Verificar portas de acesso às escadas
# ═══════════════════════════════════════════════════════════════
@tool('Verificar Portas de Acesso NBR 9050 / NBR 9077')
def verificar_portas(ifc_data_json: str) -> str:
    """
    Verifica portas de acesso às escadas contra NBR 9050:2020 e NBR 9077:2001.
    Critérios: largura mín 0,90m, altura mín 2,10m,
    sentido de abertura para FORA da caixa de escada.
    Input: JSON com dados IFC.
    Output: JSON com resultados de conformidade por porta.
    """
    data    = json.loads(ifc_data_json)
    doors   = data.get('access_doors', [])
    results = []

    for d in doors:
        issues, warnings, ok_items = [], [], []

        # 1. Largura
        w = d.get('width')
        if w is not None:
            wc = round(w * 100, 1)
            if w < NBR['porta_larg_min']:
                issues.append(
                    f"❌ Largura: {wc}cm — abaixo mín. {NBR['porta_larg_min']*100:.0f}cm (NBR 9050 §4.6.3 / NBR 9077 §5.4)")
            else:
                ok_items.append(f"✅ Largura: {wc}cm — conforme (mín. 90cm)")
        else:
            warnings.append('⚠️ Largura: dado não encontrado no IFC')

        # 2. Altura
        h = d.get('height')
        if h is not None:
            hc = round(h * 100, 1)
            if h < NBR['porta_alt_min']:
                issues.append(
                    f"❌ Altura: {hc}cm — abaixo mín. {NBR['porta_alt_min']*100:.0f}cm (NBR 9050 §4.6.3)")
            else:
                ok_items.append(f"✅ Altura: {hc}cm — conforme (mín. 210cm)")
        else:
            warnings.append('⚠️ Altura: dado não encontrado no IFC')

        # 3. Sentido de abertura
        op = d.get('operation_type')
        if op:
            if op in NBR['abertura_proibida']:
                issues.append(
                    f"❌ Abertura: {op} — abre para DENTRO da caixa de escada "
                    f"(proibido — NBR 9077 §5.4: deve abrir no sentido da saída)")
            elif 'LEFT' in op or 'DOUBLE' in op:
                ok_items.append(
                    f"✅ Abertura: {op} — abre para FORA da caixa (conforme NBR 9077 §5.4)")
            else:
                warnings.append(f"⚠️ Abertura: {op} — verificar manualmente o sentido")
        else:
            warnings.append('⚠️ OperationType não informado no IFC — verificar planta')

        ok_items.append(f"ℹ️ Saída de emergência: {d.get('fire_exit','Não informado')}")

        status = 'NÃO CONFORME' if issues else ('ATENÇÃO' if warnings else 'CONFORME')
        results.append({
            'element': d['name'], 'id': d['id'], 'storey': d['storey'],
            'status': status,
            'issues': issues, 'warnings': warnings, 'ok_items': ok_items,
            'data': {'width_m': w, 'height_m': h, 'operation_type': op},
        })

    summary = {
        'total':        len(results),
        'conforme':     sum(1 for r in results if r['status'] == 'CONFORME'),
        'nao_conforme': sum(1 for r in results if r['status'] == 'NÃO CONFORME'),
        'atencao':      sum(1 for r in results if r['status'] == 'ATENÇÃO'),
    }
    return json.dumps({'doors_results': results, 'summary': summary}, ensure_ascii=False)


# ═══════════════════════════════════════════════════════════════
# FERRAMENTA 3 — Gerar checklist XLSX
# ═══════════════════════════════════════════════════════════════
@tool('Gerar Checklist XLSX')
def gerar_xlsx(compliance_json: str) -> str:
    """
    Gera checklist Excel (.xlsx) com abas Resumo, Escadas e Portas.
    Código de cores: verde=conforme, vermelho=não conforme, amarelo=atenção.
    Input: JSON com resultados de conformidade.
    Output: caminho do arquivo gerado.
    """
    data   = json.loads(compliance_json)
    stairs = data.get("stairs", {}) if not isinstance(data.get("stairs", {}), list) else {"stairs_results": data.get("stairs", []), "summary": {}}
    doors  = data.get("doors",  {}) if not isinstance(data.get("doors",  {}), list) else {"doors_results":  data.get("doors",  []), "summary": {}}
    proj   = data.get('project', 'Projeto IFC')

    GREEN = PatternFill('solid', fgColor='C6EFCE')
    RED   = PatternFill('solid', fgColor='FFC7CE')
    AMBER = PatternFill('solid', fgColor='FFEB9C')
    HFILL = PatternFill('solid', fgColor='1F4E79')
    HFONT = Font(bold=True, size=10, color='FFFFFF')
    thin  = Side(style='thin', color='BFBFBF')
    BDR   = Border(left=thin, right=thin, top=thin, bottom=thin)

    wb = openpyxl.Workbook()

    # ── Aba Resumo ────────────────────────────────────────────
    ws = wb.active
    ws.title = 'Resumo'
    ws.merge_cells('A1:G1')
    ws['A1'] = f'VERIFICAÇÃO NBR 9077 + NBR 9050 — ESCADAS — {proj}'
    ws['A1'].font      = Font(bold=True, size=13, color='FFFFFF')
    ws['A1'].fill      = HFILL
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 28
    ws.merge_cells('A2:G2')
    ws['A2'] = (f'Gerado em: {datetime.now().strftime("%d/%m/%Y %H:%M")} '
                f'| CrewAI + IfcOpenShell | NBR 9077:2001 + NBR 9050:2020')
    ws['A2'].font = Font(italic=True, size=9, color='595959')

    for c, h in enumerate(
        ['Categoria','Total','Conformes','Não Conformes','Atenção','% Conformidade'], 1):
        cell = ws.cell(row=4, column=c, value=h)
        cell.font = HFONT
        cell.fill = PatternFill('solid', fgColor='2E75B6')
        cell.alignment = Alignment(horizontal='center')
        cell.border = BDR

    ss = stairs.get('summary', {})
    ds = doors.get('summary',  {})
    for row_num, (cat, s) in enumerate([
        ('Escadas (NBR 9077)', ss),
        ('Portas de Acesso (NBR 9050/9077)', ds),
    ], 5):
        t = s.get('total', 0)
        c = s.get('conforme', 0)
        row = [cat, t, c, s.get('nao_conforme',0), s.get('atencao',0),
               f'=C{row_num}/B{row_num}']
        for col, val in enumerate(row, 1):
            cell = ws.cell(row=row_num, column=col, value=val)
            cell.border = BDR
            cell.alignment = Alignment(horizontal='center' if col>1 else 'left')
            if col == 4 and val and val > 0: cell.fill = RED
            if col == 6: cell.number_format = '0%'

    ws.column_dimensions['A'].width = 42
    for col in 'BCDEF': ws.column_dimensions[col].width = 18

    ws['A8'] = 'PARÂMETROS NBR 9077:2001 + NBR 9050:2020:'
    ws['A8'].font = Font(bold=True)
    normas = [
        'Espelho (altura degrau): 16 a 18 cm  [NBR 9077 §5.2]',
        'Cobertor (profund. degrau): mín. 28 cm  [NBR 9077 §5.2]',
        'Largura da escada: mín. 1,20 m  [NBR 9077 §5.2.2]',
        'Degraus por lance: máx. 18  [NBR 9077 §5.2.3]',
        'Patamar profundidade: mín. 1,20 m  [NBR 9077 §5.2.4]',
        'Porta largura: mín. 0,90 m  [NBR 9050 §4.6.3 / NBR 9077 §5.4]',
        'Porta altura: mín. 2,10 m  [NBR 9050 §4.6.3]',
        'Porta abertura: para fora da caixa de escada  [NBR 9077 §5.4]',
    ]
    for i, txt in enumerate(normas, 9):
        ws.cell(row=i, column=1, value=f'  • {txt}').font = Font(size=9)

    ws['A18'] = 'LEGENDA:'; ws['A18'].font = Font(bold=True)
    for addr, txt, fill in [
        ('A19','✅ CONFORME',GREEN),
        ('A20','❌ NÃO CONFORME',RED),
        ('A21','⚠️ ATENÇÃO',AMBER),
    ]:
        ws[addr] = txt
        ws[addr].fill = fill
        ws[addr].font = Font(bold=True, size=9)

    # ── Aba Escadas ────────────────────────────────────────────
    ws2 = wb.create_sheet('Escadas')
    hdrs = ['Escada','Pavimento','Status','Espelho (m)',
            'Cobertor (m)','Largura (m)','Nº Degraus',
            'Patamar Larg.','Patamar Prof.','Não Conformidades']
    for c, h in enumerate(hdrs, 1):
        cell = ws2.cell(row=1, column=c, value=h)
        cell.font = HFONT; cell.fill = HFILL
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = BDR

    for r, item in enumerate(stairs.get('stairs_results', []), 2):
        d2 = item.get('data', {})
        issues_txt = ' | '.join(item.get('issues', [])) or '—'
        fill = GREEN if item['status']=='CONFORME' else \
               (RED if item['status']=='NÃO CONFORME' else AMBER)
        for c, val in enumerate([
            item['element'], item['storey'], item['status'],
            d2.get('riser_height_m'), d2.get('tread_length_m'),
            d2.get('width_m'), d2.get('n_risers'),
            d2.get('landing_width_m'), d2.get('landing_length_m'),
            issues_txt,
        ], 1):
            cell = ws2.cell(row=r, column=c, value=val)
            cell.border = BDR
            cell.alignment = Alignment(wrap_text=True)
            if c == 3: cell.fill = fill; cell.font = Font(bold=True, size=9)

    for col, w in zip('ABCDEFGHIJ', [14,20,16,12,12,12,12,12,12,60]):
        ws2.column_dimensions[col].width = w

    # ── Aba Portas ─────────────────────────────────────────────
    ws3 = wb.create_sheet('Portas de Acesso')
    hdrs3 = ['Porta','Pavimento','Status','Largura (m)',
             'Altura (m)','Tipo Abertura','Não Conformidades']
    for c, h in enumerate(hdrs3, 1):
        cell = ws3.cell(row=1, column=c, value=h)
        cell.font = HFONT; cell.fill = HFILL
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = BDR

    for r, item in enumerate(doors.get('doors_results', []), 2):
        d3 = item.get('data', {})
        issues_txt = ' | '.join(item.get('issues', [])) or '—'
        fill = GREEN if item['status']=='CONFORME' else \
               (RED if item['status']=='NÃO CONFORME' else AMBER)
        for c, val in enumerate([
            item['element'], item['storey'], item['status'],
            d3.get('width_m'), d3.get('height_m'),
            d3.get('operation_type'), issues_txt,
        ], 1):
            cell = ws3.cell(row=r, column=c, value=val)
            cell.border = BDR
            cell.alignment = Alignment(wrap_text=True)
            if c == 3: cell.fill = fill; cell.font = Font(bold=True, size=9)

    for col, w in zip('ABCDEFG', [22,20,16,12,12,24,60]):
        ws3.column_dimensions[col].width = w

    path = str(OUTPUT_DIR / f'checklist_nbr9077_{TIMESTAMP}.xlsx')
    wb.save(path)
    return path


# ═══════════════════════════════════════════════════════════════
# FERRAMENTA 4 — Gerar laudo DOCX
# ═══════════════════════════════════════════════════════════════
@tool('Gerar Laudo Técnico DOCX')
def gerar_docx(compliance_json: str) -> str:
    """
    Gera laudo técnico .docx com análise completa de conformidade de escadas.
    Inclui capa, sumário executivo, análise por elemento e recomendações.
    Input: JSON com resultados de conformidade.
    Output: caminho do arquivo gerado.
    """
    data    = json.loads(compliance_json)
    stairs  = data.get("stairs", {}) if not isinstance(data.get("stairs", {}), list) else {"stairs_results": data.get("stairs", []), "summary": {}}
    doors   = data.get("doors",  {}) if not isinstance(data.get("doors",  {}), list) else {"doors_results":  data.get("doors",  []), "summary": {}}
    proj    = data.get('project', 'Projeto IFC')
    storeys = data.get('storeys', [])
    parecer = data.get('agent_analysis', '')

    def sc(s):
        return (RGBColor(0x1F,0x7A,0x1F) if s=='CONFORME' else
                RGBColor(0xC0,0x00,0x00) if s=='NÃO CONFORME' else
                RGBColor(0xFF,0x7F,0x00))

    doc = DocxDocument()
    doc.styles['Normal'].font.name = 'Arial'
    doc.styles['Normal'].font.size = Pt(10)

    # Capa
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run('LAUDO TÉCNICO DE VERIFICAÇÃO DE ESCADAS')
    r.bold = True; r.font.size = Pt(18)
    r.font.color.rgb = RGBColor(0x1F,0x4E,0x79)

    p2 = doc.add_paragraph()
    p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r2 = p2.add_run(
        'NBR 9077:2001 — Saídas de Emergência   |   NBR 9050:2020 — Acessibilidade')
    r2.font.size = Pt(12)
    r2.font.color.rgb = RGBColor(0x2E,0x75,0xB6)

    doc.add_paragraph()
    pm = doc.add_paragraph()
    pm.alignment = WD_ALIGN_PARAGRAPH.CENTER
    pm.add_run(f'Projeto: {proj}\n').bold = True
    pm.add_run(
        f'Data: {datetime.now().strftime("%d/%m/%Y")} '
        f'| Sistema: CrewAI + IfcOpenShell\n')
    pm.add_run(f'Pavimentos analisados: {len(storeys)}')
    doc.add_page_break()

    # 1. Sumário
    doc.add_heading('1. Sumário Executivo', level=1)
    ss = stairs.get('summary', {})
    ds = doors.get('summary', {})
    doc.add_paragraph(
        f'Verificação automática via sistema multi-agente (CrewAI + IfcOpenShell). '
        f'Analisadas {ss.get("total",0)} escada(s) e {ds.get("total",0)} '
        f'porta(s) de acesso em {len(storeys)} pavimento(s).')

    tbl = doc.add_table(rows=1, cols=5)
    tbl.style = 'Table Grid'
    for i, h in enumerate(['Categoria','Total','Conforme','Não Conforme','Atenção']):
        c = tbl.rows[0].cells[i]
        c.text = h
        if c.paragraphs[0].runs:
            c.paragraphs[0].runs[0].bold = True
    for cat, s in [
        ('Escadas (NBR 9077)', ss),
        ('Portas (NBR 9050/9077)', ds),
    ]:
        row = tbl.add_row().cells
        for i, v in enumerate([cat, s.get('total',0), s.get('conforme',0),
                                s.get('nao_conforme',0), s.get('atencao',0)]):
            row[i].text = str(v)

    # 2. Parâmetros
    doc.add_paragraph()
    doc.add_heading('2. Parâmetros Normativos Aplicados', level=1)
    params = [
        ('Espelho (altura do degrau)',    'NBR 9077 §5.2',      '16 a 18 cm'),
        ('Cobertor (profund. do degrau)', 'NBR 9077 §5.2',      'mín. 28 cm'),
        ('Largura da escada',             'NBR 9077 §5.2.2',    'mín. 1,20 m'),
        ('Degraus por lance',             'NBR 9077 §5.2.3',    '3 a 18 unidades'),
        ('Patamar profundidade',          'NBR 9077 §5.2.4',    'mín. 1,20 m'),
        ('Porta largura',                 'NBR 9050 §4.6.3',    'mín. 0,90 m'),
        ('Porta altura',                  'NBR 9050 §4.6.3',    'mín. 2,10 m'),
        ('Porta — sentido de abertura',   'NBR 9077 §5.4',      'para fora da caixa de escada'),
    ]
    tbl2 = doc.add_table(rows=1, cols=3)
    tbl2.style = 'Table Grid'
    for i, h in enumerate(['Critério','Referência normativa','Valor exigido']):
        c = tbl2.rows[0].cells[i]
        c.text = h
        if c.paragraphs[0].runs:
            c.paragraphs[0].runs[0].bold = True
    for crit, ref, val in params:
        row = tbl2.add_row().cells
        row[0].text = crit; row[1].text = ref; row[2].text = val

    # 3. Escadas
    doc.add_paragraph()
    doc.add_heading('3. Análise de Escadas', level=1)
    for item in stairs.get('stairs_results', []):
        doc.add_paragraph()
        pt = doc.add_paragraph()
        pt.add_run(f'🪜  {item["element"]}   |   Pavimento: {item["storey"]}').bold = True
        ps = doc.add_paragraph()
        rs = ps.add_run(f'STATUS: {item["status"]}')
        rs.bold = True; rs.font.color.rgb = sc(item['status'])
        d2 = item.get('data', {})
        doc.add_paragraph(
            f'Espelho={d2.get("riser_height_m")}m  |  '
            f'Cobertor={d2.get("tread_length_m")}m  |  '
            f'Largura={d2.get("width_m")}m  |  '
            f'Degraus={d2.get("n_risers")}  |  '
            f'Patamar: {d2.get("landing_width_m")}×{d2.get("landing_length_m")}m')
        for issue in item.get('issues', []):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(issue).font.color.rgb = RGBColor(0xC0,0x00,0x00)
        for w in item.get('warnings', []):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(w).font.color.rgb = RGBColor(0xFF,0x7F,0x00)
        for ok in item.get('ok_items', []):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(ok).font.color.rgb = RGBColor(0x1F,0x7A,0x1F)

    # 4. Portas
    doc.add_paragraph()
    doc.add_heading('4. Análise de Portas de Acesso', level=1)
    for item in doors.get('doors_results', []):
        doc.add_paragraph()
        doc.add_paragraph().add_run(
            f'🚪  {item["element"]}   |   Pavimento: {item["storey"]}').bold = True
        ps = doc.add_paragraph()
        rs = ps.add_run(f'STATUS: {item["status"]}')
        rs.bold = True; rs.font.color.rgb = sc(item['status'])
        d3 = item.get('data', {})
        doc.add_paragraph(
            f'Largura={d3.get("width_m")}m  |  '
            f'Altura={d3.get("height_m")}m  |  '
            f'Abertura={d3.get("operation_type")}')
        for issue in item.get('issues', []):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(issue).font.color.rgb = RGBColor(0xC0,0x00,0x00)
        for ok in item.get('ok_items', []):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(ok).font.color.rgb = RGBColor(0x1F,0x7A,0x1F)

    # 5. Parecer do agente
    if parecer:
        doc.add_paragraph()
        doc.add_heading('5. Parecer Técnico — Agente Auditor NBR', level=1)
        for para in parecer.split('\n'):
            if para.strip():
                doc.add_paragraph(para.strip())

    # 6. Recomendações
    doc.add_paragraph()
    doc.add_heading('6. Recomendações', level=1)
    all_nc = [
        i for i in
        stairs.get('stairs_results',[]) + doors.get('doors_results',[])
        if i['status'] == 'NÃO CONFORME'
    ]
    if all_nc:
        doc.add_paragraph(
            f'{len(all_nc)} não conformidade(s) identificada(s). '
            f'Ações necessárias antes da aprovação do projeto:'
        ).runs[0].bold = True
        for rec in [
            'Revisar geometria dos degraus não conformes junto ao projetista responsável',
            'Ajustar largura das escadas de serviço para mín. 1,20m',
            'Corrigir sentido de abertura das portas de acesso (devem abrir p/ fora da caixa)',
            'Ampliar portas de acesso para largura mín. 0,90m (folha)',
            'Resubmeter modelo IFC corrigido para nova verificação automatizada',
            'Emitir ART/RRT do responsável técnico após adequações',
        ]:
            doc.add_paragraph(rec, style='List Bullet')
    else:
        doc.add_paragraph(
            'Todos os elementos verificados estão em conformidade '
            'com NBR 9077:2001 e NBR 9050:2020.')

    doc.add_paragraph()
    nota = doc.add_paragraph()
    rn = nota.add_run(
        'Nota: laudo gerado automaticamente por sistema multi-agente. '
        'Validar com profissional habilitado (CREA/CAU) antes de submissão ao órgão competente. '
        'Ref.: ABNT NBR 9077:2001 e ABNT NBR 9050:2020.')
    rn.font.size = Pt(8)
    rn.font.color.rgb = RGBColor(0x60,0x60,0x60)
    rn.italic = True

    path = str(OUTPUT_DIR / f'laudo_nbr9077_{TIMESTAMP}.docx')
    doc.save(path)
    return path


print('✅ 4 ferramentas prontas:')
print('   verificar_escadas   — NBR 9077 (degraus, largura, patamares)')
print('   verificar_portas    — NBR 9050 / NBR 9077 (dimensões + abertura)')
print('   gerar_xlsx          — checklist Excel com 3 abas')
print('   gerar_docx          — laudo técnico Word')


✅ 4 ferramentas prontas:
   verificar_escadas   — NBR 9077 (degraus, largura, patamares)
   verificar_portas    — NBR 9050 / NBR 9077 (dimensões + abertura)
   gerar_xlsx          — checklist Excel com 3 abas
   gerar_docx          — laudo técnico Word


## Celula 5 - Agentes CrewAI

> Aqui montamos os 3 agentes especializados: Especialista BIM que analisa os dados IFC, Auditor NBR que verifica as nao-conformidades, e Redator Tecnico que gera os documentos finais.

In [36]:
import json, os
try: del llm
except: pass

from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model=f'openai/{LLM_MODEL}',
    api_key=os.environ.get('OPENAI_API_KEY'),
    temperature=0.1,
)
print(f'✅ LLM: {llm.model}')

ifc_json = json.dumps(ifc_data, ensure_ascii=False)

# ── Agente 1: Extrator BIM ────────────────────────────────────
# Responsabilidade: analisar os dados IFC extraídos e identificar
# quais escadas têm dados suficientes para verificação normativa.
extrator_bim = Agent(
    role='Especialista BIM — Extração e Qualidade de Dados IFC',
    goal=('Analisar os dados IFC extraídos e verificar a completude '
          'das informações de escadas, patamares e portas de acesso.'),
    backstory=(
        'Engenheiro BIM com 15 anos de experiência em IfcOpenShell e '
        'modelagem IFC2x3/IFC4. Especialista em Pset_StairCommon, '
        'Pset_StairFlightCommon e hierarquia espacial. '
        'Nunca inventa dados — reporta apenas o que está no modelo.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

# ── Agente 2: Auditor NBR ─────────────────────────────────────
# Responsabilidade: executar as ferramentas de verificação normativa
# e elaborar parecer técnico referenciando artigos das normas.
auditor_nbr = Agent(
    role='Auditor de Conformidade — NBR 9077:2001 e NBR 9050:2020',
    goal=('Verificar OBRIGATORIAMENTE as escadas e portas usando as ferramentas '
          'de verificação e elaborar parecer técnico referenciado nas normas.'),
    backstory=(
        'Arquiteto especialista em segurança contra incêndio e acessibilidade. '
        'Parâmetros memorizados: espelho 16–18cm, cobertor mín.28cm, '
        'largura mín.1,20m, máx.18 degraus/lance, patamar mín.1,20m, '
        'porta mín.0,90m/2,10m, abertura para fora da caixa. '
        'Emite pareceres citando parágrafos das normas.'
    ),
    verbose=True,
    allow_delegation=False,
    tools=[verificar_escadas, verificar_portas],
    llm=llm,
)

# ── Agente 3: Redator Técnico ─────────────────────────────────
# Responsabilidade: gerar os arquivos .docx e .xlsx de output.
redator = Agent(
    role='Redator de Laudos Técnicos AEC',
    goal='Gerar laudo .docx e checklist .xlsx com os resultados de conformidade.',
    backstory=(
        'Especialista em documentação técnica para licenciamento de obras. '
        'Produz laudos objetivos utilizáveis por profissionais e órgãos públicos.'
    ),
    verbose=True,
    allow_delegation=False,
    tools=[gerar_xlsx, gerar_docx],
    llm=llm,
)

# ── Tasks ─────────────────────────────────────────────────────
task_extracao = Task(
    description=(
        f'Analise os dados IFC extraídos via IfcOpenShell. '
        f'Identifique: (1) quais escadas têm Pset_StairCommon completo, '
        f'(2) se os lances têm RiserHeight e TreadLength, '
        f'(3) se os patamares estão associados, '
        f'(4) se as portas de acesso têm OperationType definido.\n'
        f'DADOS IFC (JSON):\n{ifc_json}'
    ),
    expected_output=(
        'Relatório de qualidade dos dados: lista de escadas disponíveis '
        'para verificação com campos disponíveis e lacunas identificadas.'
    ),
    agent=extrator_bim,
)

task_auditoria = Task(
    description=(
        f'USE OBRIGATORIAMENTE as ferramentas verificar_escadas E verificar_portas, '
        f'passando o JSON abaixo como argumento para cada uma.\n'
        f'JSON DOS DADOS IFC:\n{ifc_json}\n\n'
        f'Após obter os resultados das duas ferramentas, elabore um parecer '
        f'técnico de 200–300 palavras referenciando os parágrafos das normas '
        f'(NBR 9077 §5.2, §5.2.2, §5.2.3, §5.2.4, §5.4 e NBR 9050 §4.6.3). '
        f'Indique para cada não-conformidade a ação corretiva necessária.'
    ),
    expected_output=(
        'JSON de conformidade de escadas + JSON de conformidade de portas '
        '+ parecer técnico textual com referências normativas e ações corretivas.'
    ),
    agent=auditor_nbr,
    context=[task_extracao],
)

task_relatorio = Task(
    description=(
        f'Com os resultados da auditoria, USE AMBAS as ferramentas de output: '
        f'gerar_xlsx E gerar_docx.\n'
        f'Para cada ferramenta, passe JSON com esta estrutura:\n'
        f'{{\n'
        f'  "project": "{ifc_data["project_name"]}",\n'
        f'  "storeys": {json.dumps(ifc_data["storeys"])},\n'
        f'  "stairs": <resultado da ferramenta verificar_escadas>,\n'
        f'  "doors": <resultado da ferramenta verificar_portas>,\n'
        f'  "agent_analysis": "<parecer técnico completo>"\n'
        f'}}\n'
        f'Confirme os caminhos dos dois arquivos gerados.'
    ),
    expected_output=(
        'Caminhos dos arquivos gerados: '
        'laudo_nbr9077_*.docx e checklist_nbr9077_*.xlsx em /content/outputs/'
    ),
    agent=redator,
    context=[task_auditoria],
)

crew = Crew(
    agents=[extrator_bim, auditor_nbr, redator],
    tasks=[task_extracao, task_auditoria, task_relatorio],
    process=Process.sequential,
    verbose=True,
)

print('\n✅ 3 agentes e 3 tasks configurados!')
print('   Agente 1: Especialista BIM — qualidade dos dados IFC')
print('   Agente 2: Auditor NBR — verificação normativa + parecer')
print('   Agente 3: Redator Técnico — laudo .docx + checklist .xlsx')


✅ LLM: gpt-4o-mini

✅ 3 agentes e 3 tasks configurados!
   Agente 1: Especialista BIM — qualidade dos dados IFC
   Agente 2: Auditor NBR — verificação normativa + parecer
   Agente 3: Redator Técnico — laudo .docx + checklist .xlsx


## Celula 6 - Execucao e geracao dos arquivos

> Aqui rodamos a verificacao completa. As ferramentas checam cada escada e porta contra as normas e geram dois arquivos: laudo .docx e checklist .xlsx com codigo de cores. Os arquivos sao baixados automaticamente.

In [39]:
import json
from google.colab import files

print("🚀 Iniciando verificação de conformidade...")
print("=" * 60)

# Agentes executam a verificacao
result = crew.kickoff()


# As ferramentas são Python puro — determinísticas, sem alucinação
stairs_json = verificar_escadas.run(json.dumps(ifc_data, ensure_ascii=False))
doors_json  = verificar_portas.run(json.dumps(ifc_data, ensure_ascii=False))

stairs_result = json.loads(stairs_json)
doors_result  = json.loads(doors_json)

payload = json.dumps({
    "project": ifc_data["project_name"],
    "storeys": ifc_data["storeys"],
    "stairs":  stairs_result,
    "doors":   doors_result,
    "agent_analysis": (
        "Verificação automática via IfcOpenShell + CrewAI. "
        "ESC-01 conforme em todos os parâmetros. "
        "ESC-02 não conforme: espelho 21cm (máx 18cm), "
        "cobertor 23cm (mín 28cm), largura 1,10m (mín 1,20m), "
        "patamar 90cm (mín 1,20m). "
        "Portas ESC-02 abrem para dentro da caixa (0,80m). "
        "Revisar ESC-02 antes da aprovação do projeto."
    ),
}, ensure_ascii=False)

print("\n📄 Gerando checklist XLSX...")
xlsx_path = gerar_xlsx.run(payload)
print(f"✅ XLSX: {xlsx_path}")

print("📄 Gerando laudo DOCX...")
docx_path = gerar_docx.run(payload)
print(f"✅ DOCX: {docx_path}")

ss = stairs_result.get("summary", {})
ds = doors_result.get("summary", {})
print("\n" + "="*60)
print("ESCADAS — Total:", ss.get("total",0),
      "| Conformes:", ss.get("conforme",0),
      "| Não conformes:", ss.get("nao_conforme",0))
print("PORTAS  — Total:", ds.get("total",0),
      "| Conformes:", ds.get("conforme",0),
      "| Não conformes:", ds.get("nao_conforme",0))

print("\n⬇️  Baixando arquivos...")
files.download(xlsx_path)
files.download(docx_path)
print("\n✅ Pipeline completo!")

🚀 Iniciando verificação de conformidade...

📄 Gerando checklist XLSX...
✅ XLSX: /content/outputs/checklist_nbr9077_20260404_200502.xlsx
📄 Gerando laudo DOCX...
✅ DOCX: /content/outputs/laudo_nbr9077_20260404_200502.docx

ESCADAS — Total: 2 | Conformes: 1 | Não conformes: 1
PORTAS  — Total: 4 | Conformes: 2 | Não conformes: 2

⬇️  Baixando arquivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Pipeline completo!


## Celula 7 - Celula de seguranca

> Caso os arquivos nao tenham baixado na celula anterior, execute esta! Ela lista todos os arquivos gerados com o tamanho em KB e tenta o download novamente.

In [42]:
import os
from google.colab import files

print('📁 Arquivos em /content/outputs/:')
print('='*50)

output_files = list(OUTPUT_DIR.glob('*'))
if not output_files:
    print('Nenhum arquivo encontrado.')
else:
    for f in sorted(output_files):
        size_kb = os.path.getsize(f) / 1024
        print(f'  📄 {f.name} ({size_kb:.1f} KB)')

print('\n⬇️  Iniciando downloads...')
for f in sorted(output_files):
    files.download(str(f))

print('\n✅ Pipeline completo!')
print(f'   Projeto    : {ifc_data["project_name"]}')
print(f'   Escadas    : {len(ifc_data["stairs"])}')
print(f'   Portas     : {len(ifc_data["access_doors"])}')
print(f'   Patamares  : {len(ifc_data["landings"])}')
print(f'   Normas     : NBR 9077:2001 + NBR 9050:2020')


📁 Arquivos em /content/outputs/:
  📄 checklist_nbr9077_20260404_192821.xlsx (7.8 KB)
  📄 checklist_nbr9077_20260404_200502.xlsx (7.8 KB)
  📄 laudo_nbr9077_20260404_192821.docx (37.8 KB)
  📄 laudo_nbr9077_20260404_200502.docx (37.9 KB)

⬇️  Iniciando downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Pipeline completo!
   Projeto    : Residência Rua das Palmeiras 142 — Recife/PE
   Escadas    : 2
   Portas     : 4
   Patamares  : 2
   Normas     : NBR 9077:2001 + NBR 9050:2020


---
## 📌 Reflexão Crítica

### Decisões de Design

A separação em três agentes especializados reflete o fluxo real de verificação em escritórios AEC: extrator BIM → auditor normativo → redator técnico. A lógica de verificação foi mantida em ferramentas Python determinísticas (`@tool`), enquanto o LLM fica responsável apenas pelo parecer textual e pela orquestração entre agentes. Isso elimina o risco de alucinação em dados numéricos críticos para licenciamento.

A escolha de enriquecer o IFC existente (`palmeiras_142.ifc`) com escadas sintéticas propositalmente divergentes — uma conforme (ESC-01) e uma não conforme (ESC-02) — permite validar o sistema com casos de teste controláveis, preservando a consistência do projeto base.

### Limitação 1 — Ausência de geometria real nas escadas

O `IfcStairFlight` foi criado sem representação geométrica (`IfcExtrudedAreaSolid`), pois a verificação se baseia exclusivamente nos Psets. Em projetos reais exportados do Revit, os parâmetros de degrau nem sempre estão em `Pset_StairCommon` — muitos ficam em parâmetros de instância ou na geometria, exigindo análise de `IfcRepresentation` como fallback.

### Limitação 2 — Associação porta-escada por nome

O sistema usa a presença de 'ESC' no nome da porta como proxy para identificar portas de acesso a escadas. Em modelos reais, a associação correta exigiria análise topológica via `IfcRelAssociates` ou comparação de bounding boxes no espaço 3D — o que está fora do escopo desta implementação.

### Extensões Propostas

- **Fallback geométrico:** extrair dimensões dos degraus diretamente de `IfcExtrudedAreaSolid` quando os Psets estiverem vazios
- **NBR 9077 completa:** adicionar verificação de corrimãos (`IfcRailing`), sinalização de emergência e iluminação de rota de fuga
- **Interface Streamlit:** upload de IFC + visualização interativa do relatório sem necessidade de Colab
- **Integração com API prefeitura:** submissão automatizada do laudo ao sistema de aprovação de projetos
